In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from tqdm import tqdm
from openbabel import pybel
from rdkit.Chem.MolStandardize import rdMolStandardize
# from molvs import Standardizer
from tqdm import tqdm
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import MolStandardize
from rdkit.Chem.MolStandardize import rdMolStandardize
import sys
sys.path.append('..')
from metrics import cal_metrics
RDLogger.DisableLog('rdApp.*')
from rdkit.Chem import QED
import metrics.SA_Score.sascorer as sascorer

tqdm.pandas()

In [ ]:
# 1. smiles
def is_valid(smiles):
    if smiles is None:
        return None
    try:
        mol = Chem.MolFromSmiles(smiles)
    except ValueError:
        return None
    if smiles != '' and mol is not None and mol.GetNumAtoms() > 0:
        try:
            Chem.SanitizeMol(mol)
        except ValueError:
            return None
        return  Chem.MolToSmiles(mol)

    return None

In [ ]:
# 2. desalting
def strip_salt(smi):
    try:
        mol = pybel.readstring("smi", smi)  # Read as mol in smiles format
        # strip salt
        mol.OBMol.StripSalts(10)
        mols = mol.OBMol.Separate()
        mol = pybel.Molecule(mols[0])
        for imol in mols:
            imol = pybel.Molecule(imol)
            if len(imol.atoms) > len(mol.atoms):
                mol = imol
        smi_clean = mol.write('smi')
        smi_clean = smi_clean.replace('\n', '')
        smi_clean = smi_clean.split()[0]
        return smi_clean
    except Exception as e:
        # print(f"Error processing SMILES: {smi}. Exception: {e}")
        return None

In [ ]:
# 3. Standardization, de-stereotyping
def smiles_cleaning(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
    except Exception as e:
        return None
    if mol:
        try:
            mol=rdMolStandardize.Normalize(mol)
            smi=Chem.MolToSmiles(mol, isomericSmiles=False, canonical=True)
            return smi
        except Exception as e:
            return None
    else:
        return None


In [ ]:
# 4. Wash again
# Wash smiles, remove salt, and make them three-dimensional
class MolCleaner(object):
    def __init__(self):
        self.normarizer = MolStandardize.normalize.Normalizer()
        self.lfc = MolStandardize.fragment.LargestFragmentChooser()
        self.uc = MolStandardize.charge.Uncharger()

    def process(self, smi):
        mol = Chem.MolFromSmiles(smi)
        if mol:
            mol = self.normarizer.normalize(mol)
            mol = self.lfc.choose(mol)
            mol = self.uc.uncharge(mol)
            smi = Chem.MolToSmiles(mol, isomericSmiles=False, canonical=True)
            return smi
        else:
            return None

In [ ]:
# 5 Calculate the number of rings
def cal_SSSR(smi):
    mol = Chem.MolFromSmiles(smi)
    sssr = Chem.GetSSSR(mol)
    return len(sssr)

In [ ]:
# 6 Calculate the number of atoms in the ring that is less than 6
def ring_filter(smi):
    mol = Chem.MolFromSmiles(smi)
    # has_ring = mol.GetNumBonds() - mol.GetNumAtoms() + 1 > 0
    # print(has_ring)

    ring_sizes = [len(ring) for ring in mol.GetRingInfo().AtomRings()]

    if ring_sizes:
        return max(ring_sizes)
    else:
        return None

In [ ]:
# 6 Calculate the number of heavy atoms
def get_num_atom(smi):
    mol = Chem.MolFromSmiles(smi)
    n = mol.GetNumHeavyAtoms()  # GetNumAtoms()
    return n

In [ ]:
# Obtain valid smiles
sm_file = r'./chembl_35_chemreps.txt'
data_all = pd.read_csv(sm_file,sep='\t')
df = data_all[['chembl_id','canonical_smiles']]


tqdm.pandas(desc="cal_val_smi")
df['valid_smiles'] = df['canonical_smiles'].progress_apply(lambda x: is_valid(x))

print(df['valid_smiles'].isna().sum())

cal_val_smi: 100%|██████████| 2474590/2474590 [10:59<00:00, 3749.91it/s]
/tmp/ipykernel_4168565/3457102980.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['valid_smiles'] = df['canonical_smiles'].progress_apply(lambda x: is_valid(x))


1


In [ ]:
# Go Salt
tqdm.pandas(desc="strip salt")
df['salted_smi'] = df['valid_smiles'].progress_apply(lambda x: strip_salt(x))

print(df['salted_smi'].isna().sum())

strip salt:  19%|█▊        | 463683/2474590 [02:00<11:52, 2822.04it/s]==============================
*** Open Babel Warning  in CreateCisTrans
  Error in cis/trans stereochemistry specified for the double bond

strip salt:  48%|████▊     | 1190451/2474590 [05:12<03:36, 5937.11it/s]==============================
*** Open Babel Warning  in CreateCisTrans
  Error in cis/trans stereochemistry specified for the double bond

strip salt:  54%|█████▍    | 1338997/2474590 [05:51<04:53, 3872.64it/s]==============================
*** Open Babel Warning  in CreateCisTrans
  Error in cis/trans stereochemistry specified for the double bond

strip salt:  70%|███████   | 1734839/2474590 [07:25<03:30, 3516.97it/s]==============================
*** Open Babel Warning  in CreateCisTrans
  Error in cis/trans stereochemistry specified for the double bond

strip salt:  70%|███████   | 1742831/2474590 [07:28<04:51, 2508.23it/s]==============================
*** Open Babel Warning  in CreateCisTrans
  Error i

1025


In [ ]:
# De-stereotyping
tqdm.pandas(desc="clean smi")
df['clean_smi'] = df['salted_smi'].progress_apply(lambda x: smiles_cleaning(x))

print(df['clean_smi'].isna().sum())

clean smi: 100%|██████████| 2474590/2474590 [22:47<00:00, 1809.63it/s]


1025


In [ ]:
# Wash again
cleaner = MolCleaner()
tqdm.pandas(desc="applying molCleaner class")
df['clean_two'] = df['clean_smi'].progress_apply(lambda x: cleaner.process(x) if pd.notnull(x) else None)

print(df['clean_two'].isna().sum())

applying molCleaner class: 100%|██████████| 2474590/2474590 [24:47<00:00, 1664.11it/s]


1025


In [ ]:

tqdm.pandas(desc="cal_val_smi_2")
df['stand_smi'] = df['clean_two'].progress_apply(lambda x: is_valid(x))
print(df['stand_smi'].isna().sum())

cal_val_smi_2: 100%|██████████| 2474590/2474590 [09:37<00:00, 4287.37it/s]


1172


In [ ]:

df_stand = df[df['stand_smi'].notnull()]
df_standD = df_stand.drop_duplicates('stand_smi')

df_standD = df_standD.reset_index(drop=True)
print(len(df_standD))

2255721


In [ ]:

df_standD = df_standD[['chembl_id', 'stand_smi', 'clean_two','clean_smi']]
df_standD.to_csv('./chembl_clean.csv', index=False)
print(len(df_standD))
print(df_standD)

2255721
             chembl_id                                          stand_smi  \
0         CHEMBL153534                       Cc1cc(-c2csc(N=C(N)N)n2)cn1C   
1         CHEMBL440060  CCC(C)C(NC(=O)C(CC(C)C)NC(=O)C(NC(=O)C(N)CCSC)...   
2         CHEMBL440245  CCCCC1NC(=O)C(NC(=O)C(CC(C)C)NC(=O)C(NC(=O)C(C...   
3         CHEMBL440249  CC(C)CC1NC(=O)CNC(=O)C(c2ccc(O)cc2)NC(=O)C(C(C...   
4         CHEMBL405398             Brc1cccc(Nc2ncnc3ccncc23)c1NCCN1CCOCC1   
...                ...                                                ...   
2255716  CHEMBL4296953                        NC(=O)c1nc2c([nH]c1=O)CCCC2   
2255717  CHEMBL4296957               CCOC(=O)C(C#N)C(=N)C1C(=O)OC2CCCCC21   
2255718  CHEMBL4298702  c1ccc(C2CC(C3CC(c4ccccc4)OC(c4ccccc4)C3)CC(c3c...   
2255719  CHEMBL4298703  CSCCC(NC=O)C(=O)NC(CCCNC(=N)NS(=O)(=O)c1c(C)c(...   
2255720  CHEMBL4298680  C[n+]1cn(C2OC(COP(=O)(S)OP(=O)(O)OP(=O)([O-])O...   

                                                 clean_two  \
0    

In [ ]:

def safe_apply(func, molecule):
    molecule = Chem.MolFromSmiles(molecule)
    try:
        return func(molecule)
    except Exception:
        return np.nan

In [27]:
df_standD = pd.read_csv('./chembl_clean.csv')
# tqdm.pandas(desc="cal qed")
# df_standD['qed'] = df_standD['stand_smi'].progress_apply(lambda x: safe_apply(QED.qed, x))

In [28]:
tqdm.pandas(desc="cal sa")
df_standD['sa'] = df_standD['stand_smi'].progress_apply(lambda x: safe_apply(sascorer.calculateScore, x))

cal sa: 100%|██████████| 2255721/2255721 [25:24<00:00, 1479.35it/s]


In [29]:
print(df_standD)

             chembl_id                                          stand_smi  \
0         CHEMBL153534                       Cc1cc(-c2csc(N=C(N)N)n2)cn1C   
1         CHEMBL440060  CCC(C)C(NC(=O)C(CC(C)C)NC(=O)C(NC(=O)C(N)CCSC)...   
2         CHEMBL440245  CCCCC1NC(=O)C(NC(=O)C(CC(C)C)NC(=O)C(NC(=O)C(C...   
3         CHEMBL440249  CC(C)CC1NC(=O)CNC(=O)C(c2ccc(O)cc2)NC(=O)C(C(C...   
4         CHEMBL405398             Brc1cccc(Nc2ncnc3ccncc23)c1NCCN1CCOCC1   
...                ...                                                ...   
2255716  CHEMBL4296953                        NC(=O)c1nc2c([nH]c1=O)CCCC2   
2255717  CHEMBL4296957               CCOC(=O)C(C#N)C(=N)C1C(=O)OC2CCCCC21   
2255718  CHEMBL4298702  c1ccc(C2CC(C3CC(c4ccccc4)OC(c4ccccc4)C3)CC(c3c...   
2255719  CHEMBL4298703  CSCCC(NC=O)C(=O)NC(CCCNC(=N)NS(=O)(=O)c1c(C)c(...   
2255720  CHEMBL4298680  C[n+]1cn(C2OC(COP(=O)(S)OP(=O)(O)OP(=O)([O-])O...   

                                                 clean_two  \
0            

In [30]:
tqdm.pandas(desc="cal sa0-1")
df_standD['sa0-1'] = df_standD['stand_smi'].progress_apply(lambda x: safe_apply(
        lambda mol: round((10 - sascorer.calculateScore(mol)) / 9, 2), x
    ))

print(df_standD['sa0-1'])

cal sa0-1: 100%|██████████| 2255721/2255721 [25:45<00:00, 1459.46it/s]

0          0.77
1          0.10
2          0.03
3          0.09
4          0.82
           ... 
2255716    0.81
2255717    0.63
2255718    0.76
2255719    0.26
2255720    0.46
Name: sa0-1, Length: 2255721, dtype: float64


In [31]:
tqdm.pandas(desc="cal qed")
df_standD['qed'] = df_standD['stand_smi'].progress_apply(lambda x: safe_apply(QED.qed, x))
print(df_standD)

cal qed: 100%|██████████| 2255721/2255721 [39:33<00:00, 950.30it/s] 

             chembl_id                                          stand_smi  \
0         CHEMBL153534                       Cc1cc(-c2csc(N=C(N)N)n2)cn1C   
1         CHEMBL440060  CCC(C)C(NC(=O)C(CC(C)C)NC(=O)C(NC(=O)C(N)CCSC)...   
2         CHEMBL440245  CCCCC1NC(=O)C(NC(=O)C(CC(C)C)NC(=O)C(NC(=O)C(C...   
3         CHEMBL440249  CC(C)CC1NC(=O)CNC(=O)C(c2ccc(O)cc2)NC(=O)C(C(C...   
4         CHEMBL405398             Brc1cccc(Nc2ncnc3ccncc23)c1NCCN1CCOCC1   
...                ...                                                ...   
2255716  CHEMBL4296953                        NC(=O)c1nc2c([nH]c1=O)CCCC2   
2255717  CHEMBL4296957               CCOC(=O)C(C#N)C(=N)C1C(=O)OC2CCCCC21   
2255718  CHEMBL4298702  c1ccc(C2CC(C3CC(c4ccccc4)OC(c4ccccc4)C3)CC(c3c...   
2255719  CHEMBL4298703  CSCCC(NC=O)C(=O)NC(CCCNC(=N)NS(=O)(=O)c1c(C)c(...   
2255720  CHEMBL4298680  C[n+]1cn(C2OC(COP(=O)(S)OP(=O)(O)OP(=O)([O-])O...   

                                                 clean_two  \
0            

In [ ]:

tqdm.pandas(desc="cal_SSSR by ss")
df_standD['ss_SSSR'] = df_standD['stand_smi'].progress_apply(lambda x: cal_SSSR(x))

cal_SSSR by ss: 100%|██████████| 2255721/2255721 [04:44<00:00, 7931.71it/s] 


In [ ]:

tqdm.pandas(desc="cal_ring by ss")
df_standD['ss_atomsInRing'] = df_standD['stand_smi'].progress_apply(lambda x: ring_filter(x))

cal_ring by ss: 100%|██████████| 2255721/2255721 [16:27<00:00, 2283.41it/s]


In [ ]:

tqdm.pandas(desc="get_heavy_atom")
df_standD['Heavy_atom'] = df_standD['stand_smi'].progress_apply(lambda x: get_num_atom(x))

get_heavy_atom: 100%|██████████| 2255721/2255721 [04:13<00:00, 8895.35it/s] 


In [ ]:

tqdm.pandas(desc="get len")
df_standD['smiles_len'] = df_standD['stand_smi'].progress_apply(lambda x: len(x))

get len: 100%|██████████| 2255721/2255721 [00:00<00:00, 2366473.41it/s]


In [ ]:

df_standD = df_standD[['chembl_id', 'stand_smi', 'qed','sa','sa0-1','ss_SSSR','ss_atomsInRing','Heavy_atom','smiles_len']]
df_standD = df_standD.dropna()
df_standD.to_csv('./chembl_clean_attribute.csv', index=False)

In [37]:
len(df_standD)

2236496

In [38]:
# QED、SA
df_cal = df_standD[(df_standD['qed'] > 0.5) & (df_standD['sa'] < 5)]
df_cal.to_csv('./chembl_clean_qed_sa.csv', index=False)
print(len(df_cal))

1318853


In [ ]:
# QED、SA、ring
df_ring = df_cal[(df_cal['ss_atomsInRing'] <= 6) & (df_cal['ss_SSSR'] <= 4)]

In [ ]:
# QED、SA、ring
df_heavy_atom = df_ring[(df_ring['Heavy_atom'] >= 10) & (df_ring['Heavy_atom'] <= 50)]
df_heavy_atom.to_csv('./chembl_clean_heavy_atom.csv', index=False)
print(len(df_heavy_atom))

1153913


In [ ]:
# QED、SA、ring、length
df_smi_len = df_ring[(df_ring['smiles_len'] >= 20) & (df_ring['smiles_len'] <= 120)]
df_smi_len.to_csv('./chembl_clean_smi_len.csv', index=False)
print(len(df_smi_len))

1144190


In [ ]:

train_smiles_list = df_smi_len['stand_smi'].tolist()
with open('./chembl_clean_smi_len.smi', 'w') as f:
    for smiles in train_smiles_list:
        f.write(smiles + '\n')
print("train set SMILES: 'moses2_train.smi' ")

训练集SMILES列已保存为 'moses2_train.smi' 文件


In [ ]:

train_smiles_list = df_heavy_atom['stand_smi'].tolist()
with open('./chembl_clean_heavy_atom.smi', 'w') as f:
    for smiles in train_smiles_list:
        f.write(smiles + '\n')
print("The training set SMILES column has been saved as the ‘chembl_heavy_atom.smi’ file.")

训练集SMILES列已保存为 'chembl_heavy_atom.smi' 文件
